# 4D - Alvos clusterizados: onde o sinal atua mais forte, em que lag, e o que passa o gate

**Pergunta especifica.** Depois da distribuicao pixel-a-pixel (4C), quais sao os alvos clusterizados mais afetados no Brasil, em que lag atua cada tipo de sinal (El Nino vs La Nina), o sinal e estavel entre subperiodos e quais variaveis do Pacifico passam o gate para as fases de modelagem?

**Metodologia.** Cada pixel vira um vetor de resposta: perfil r(lag) da SSTA Nino 3.4 nas condicoes El Nino e La Nina (lags 0-52, passo 4). K-means com k=2..8 escolhido por silhueta; clusters ranqueados por |r| maximo medio e fracao de pixels FDR-significativos. Para cada alvo: serie regional = media da anomalia padronizada dos pixels; estabilidade da correlacao no lag otimo em 1993-2009 vs 2010-presente (N_eff); gate por variavel do conjunto Pacifico: `preditor` (significativa, estavel, lag > 6 sem), `preditor_com_ressalva`, `diagnostico` (lag 0-6) ou `excluido`. Escopo estritamente Pacifico.

**Saidas.** `phase4D_clusters_pixels.csv`, `phase4D_cluster_ranking.csv`, `phase4D_cluster_lags_por_sinal.csv`, `phase4D_estabilidade.csv`, `phase4D_gate_ml.csv`, `phase4D_mapa_clusters.png`, `phase4D_perfis_clusters.png`, `phase4D_sintese_gate.png`.

In [1]:
import sys; sys.path.insert(0,'.')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import xarray as xr
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import fase4_utils as u

ds = xr.open_zarr(u.ZSTATS/'phase4C_atlas_pixel.zarr')
px = pd.read_csv(u.FEAT/'phase4_chirps_pixels.csv')
best = pd.read_csv(u.STATS/'phase4C_best_lag_pixel.csv')
SUBLAGS = [l for l in ds.lag.values if l <= 52 and l % 4 == 0]
feat = np.hstack([
    ds['r_el_nino'].sel(variavel='nino34_ssta', lag=SUBLAGS).values.T,
    ds['r_la_nina'].sel(variavel='nino34_ssta', lag=SUBLAGS).values.T,
])
ok = np.isfinite(feat).all(axis=1)
X = StandardScaler().fit_transform(feat[ok])
scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    scores[k] = silhouette_score(X, km.labels_, sample_size=min(4000, len(X)), random_state=42)
K = max(scores, key=scores.get)
km = KMeans(n_clusters=K, n_init=25, random_state=42).fit(X)
labels = np.full(len(px), -1); labels[ok] = km.labels_
print('silhueta por k:', {k: round(v,3) for k,v in scores.items()}, '| escolhido K =', K)
clusters = px.assign(cluster=labels)
u.save_table(clusters, 'phase4D_clusters_pixels.csv', index=False)

FileNotFoundError: No such file or directory: '/mnt/c/DEV/NINO26/data/processed/zarr/statistics/phase4C_atlas_pixel.zarr'

In [2]:
rows_rank, rows_lag = [], []
for c in range(K):
    m = clusters['cluster'].values == c
    for cond in ['el_nino','la_nina']:
        sub = best[(best.condicao==cond) & (best.variavel=='nino34_ssta')].set_index('pixel_id').loc[px['pixel_id']]
        bl = sub.loc[m, 'best_lag_sem'].dropna(); rr = sub.loc[m, 'r_no_best_lag'].dropna()
        rows_lag.append({'cluster': c, 'sinal': cond, 'n_pixels': int(m.sum()),
            'frac_sig_%': round(100*len(bl)/max(m.sum(),1),1),
            'lag_mediano_sem': float(bl.median()) if len(bl) else np.nan,
            'lag_iqr_sem': f"{bl.quantile(.25):.0f}-{bl.quantile(.75):.0f}" if len(bl) else '-',
            'r_mediano': round(float(rr.median()),3) if len(rr) else np.nan,
            'sentido': ('umido' if rr.median()>0 else 'seco') if len(rr) else '-'})
    perfil_en = ds['r_el_nino'].sel(variavel='nino34_ssta').values[:, m]
    rows_rank.append({'cluster': c, 'n_pixels': int(m.sum()),
        'abs_r_max_medio': round(float(np.nanmean(np.nanmax(np.abs(perfil_en), axis=0))),3),
        'lat_media': round(float(px.loc[m,'lat'].mean()),1), 'lon_media': round(float(px.loc[m,'lon'].mean()),1)})
rank = pd.DataFrame(rows_rank).sort_values('abs_r_max_medio', ascending=False)
rank['alvo_prioritario'] = range(1, len(rank)+1)
u.save_table(rank, 'phase4D_cluster_ranking.csv', index=False)
lags_tab = pd.DataFrame(rows_lag)
u.save_table(lags_tab, 'phase4D_cluster_lags_por_sinal.csv', index=False)
print(rank.to_string(index=False)); print(); print(lags_tab.to_string(index=False))

NameError: name 'K' is not defined

In [3]:
import matplotlib.cm as cm
cores = cm.get_cmap('tab10', K)
fig, ax = plt.subplots(figsize=(7.8, 8.2))
vals = np.where(clusters['cluster'].values>=0, clusters['cluster'].values, np.nan)
sc = u.pixel_map(ax, px, vals, cmap=cm.get_cmap('tab10', K), vmin=-.5, vmax=K-.5,
                 title=f'4D - Alvos clusterizados pelo perfil de resposta ENSO (K={K}, silhueta={scores[K]:.2f})')
cb = fig.colorbar(sc, ax=ax, ticks=range(K), shrink=.8); cb.set_label('cluster')
u.stamp_caption(fig, variavel='clusters k-means do perfil r(lag) EN+LN da SSTA Nino 3.4', area='Brasil 0.25', periodo='1981-2026', fonte='CHIRPS, OISST', extra='clusters derivados do sinal pixel-a-pixel do 4C; ranking em phase4D_cluster_ranking.csv')
u.save_fig(fig, 'phase4D_mapa_clusters.png')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13.8, 5), sharey=True)
for ax, cond, tit in [(axes[0],'r_el_nino','Sinal El Nino'), (axes[1],'r_la_nina','Sinal La Nina')]:
    for c in range(K):
        m = clusters['cluster'].values == c
        prof = np.nanmean(ds[cond].sel(variavel='nino34_ssta').values[:, m], axis=1)
        ax.plot(ds.lag.values, prof, color=cores(c), lw=1.9, label=f'cluster {c}')
    ax.axhline(0, color='k', lw=.6); ax.set_xlabel('lag (semanas; Pacifico antecede)')
    ax.set_title(tit, fontsize=10); ax.grid(alpha=.25)
axes[0].set_ylabel('r medio do cluster'); axes[0].legend(fontsize=8, ncol=2)
u.stamp_caption(fig, variavel='perfil medio r(lag) por cluster e tipo de sinal', area='clusters 4D', periodo='1981-2026', fonte='CHIRPS, OISST', extra='mostra em que lag cada tipo de sinal atua em cada alvo')
u.save_fig(fig, 'phase4D_perfis_clusters.png')
plt.show()

NameError: name 'K' is not defined

In [4]:
# Estabilidade por subperiodo e gate para as fases de modelagem (Pacifico-only)
Z, _px2 = u.build_chirps_weekly_zanom()
w = u.load_pacific_weekly()
Z = Z.loc[Z.index.intersection(w.index)]
alvo = pd.DataFrame({c: Z.iloc[:, np.where(clusters['cluster'].values==c)[0]].mean(axis=1) for c in range(K)})
rows = []
for c in range(K):
    for _, lrow in lags_tab[lags_tab.cluster==c].iterrows():
        L = lrow['lag_mediano_sem']
        if not np.isfinite(L):
            continue
        L = int(L); y = alvo[c]
        for v in u.PACIFIC_VARS:
            x = w[v].shift(L).reindex(y.index)
            r0, p0, ne = u.neff_corr(x, y)
            r1, p1, _ = u.neff_corr(x.loc['1993':'2009'], y.loc['1993':'2009'])
            r2, p2, _ = u.neff_corr(x.loc['2010':], y.loc['2010':])
            estavel = bool(np.isfinite(p1) and np.isfinite(p2) and p1 < .1 and p2 < .1 and np.sign(r1) == np.sign(r2))
            if not np.isfinite(r0) or p0 >= .1:
                classe = 'excluido'
            elif estavel and L > 6:
                classe = 'preditor'
            elif L <= 6:
                classe = 'diagnostico'
            else:
                classe = 'preditor_com_ressalva'
            rows.append({'cluster': c, 'sinal': lrow['sinal'], 'variavel': v, 'lag_sem': L,
                'r_total': round(r0,3) if np.isfinite(r0) else np.nan, 'p_total': round(p0,4) if np.isfinite(p0) else np.nan,
                'r_1993_2009': round(r1,3) if np.isfinite(r1) else np.nan, 'r_2010_hoje': round(r2,3) if np.isfinite(r2) else np.nan,
                'p_1993_2009': round(p1,4) if np.isfinite(p1) else np.nan, 'p_2010_hoje': round(p2,4) if np.isfinite(p2) else np.nan,
                'estavel': estavel, 'classe_gate': classe})
gate = pd.DataFrame(rows)
u.save_table(gate[['cluster','sinal','variavel','lag_sem','r_1993_2009','p_1993_2009','r_2010_hoje','p_2010_hoje','estavel']], 'phase4D_estabilidade.csv', index=False)
u.save_table(gate, 'phase4D_gate_ml.csv', index=False)
print(gate['classe_gate'].value_counts().to_string())

pv = gate.pivot_table(index=['cluster','sinal'], columns='variavel', values='r_total')
pv = pv[[v for v in u.PACIFIC_VARS if v in pv.columns]]
fig, ax = plt.subplots(figsize=(11.5, .55*len(pv)+2.6))
im = ax.imshow(pv.values, cmap='BrBG', vmin=-.6, vmax=.6, aspect='auto')
ax.set_xticks(range(pv.shape[1])); ax.set_xticklabels([u.var_label(v) for v in pv.columns], fontsize=8)
ax.set_yticks(range(len(pv))); ax.set_yticklabels([f"alvo {c} | {s}" for c, s in pv.index], fontsize=8)
sub = gate.set_index(['cluster','sinal','variavel'])
for i, (c, s) in enumerate(pv.index):
    for j, v in enumerate(pv.columns):
        try:
            g = sub.loc[(c, s, v)]
            if np.isfinite(g['r_total']):
                ax.text(j, i, f"{g['r_total']:+.2f}", ha='center', va='center', fontsize=6.6,
                        weight='bold' if g['classe_gate']=='preditor' else 'normal')
        except Exception:
            pass
ax.set_title('4D - r (N_eff) no lag otimo por alvo, sinal e variavel; negrito = passa o gate preditor')
fig.colorbar(im, ax=ax, shrink=.85, label='r no lag otimo do alvo')
u.stamp_caption(fig, variavel='correlacao Pacifico->chuva do alvo no lag otimo', area='alvos 4D', periodo='1981-2026; estabilidade 1993-2009 vs 2010+', fonte='CHIRPS, OISST, GLORYS12/UFS, ERA5', extra='gate completo em phase4D_gate_ml.csv; escopo estritamente Pacifico')
u.save_fig(fig, 'phase4D_sintese_gate.png')
plt.show()

GroupNotFoundError: group not found at path ''

**Leitura do 4D (gate G1).** Os alvos surgem dos dados (clusters do perfil de resposta pixelizado do 4C) e cada um recebe: ranking de afetacao, lag de atuacao por tipo de sinal, estabilidade entre subperiodos e o gate por variavel do Pacifico. `phase4D_gate_ml.csv` e a lista auditavel do que segue para as Fases 5B/6C como `preditor`, o que fica como `diagnostico` (lag curto) e o que e `excluido`. Sinais instaveis entre 1993-2009 e 2010-presente entram apenas com ressalva.